# REWIND ingest pipeline: motion gating → keyframes → events → index hand-off

Merged from Farid's `explore.ipynb` (v1) and Muzayyan's `rewind_pipeline_pictures.ipynb` (v2).

| Part | Code | Needs |
|---|---|---|
| Stage 1: motion gates diff, MOG2 (+ DINOv2 "deep" gate and a fused vote) | `motion_gating.py`, `deep_gate.py` | CPU (deep gate: GPU) |
| Stage 2: CPU keyframes (first / peak / change / maxgap / heartbeat) | `keyframes.py` | CPU |
| Ground-truth check (optional) | `evaluate.py` | a CSV `start_s,end_s[,label]` |
| Stage 3: events (YOLO-World + ByteTrack, SigLIP, X-CLIP) | `rewind_pipeline.py` | GPU |
| Stage 4: samplers + hand-off `index_records.jsonl` / `index_embeddings.npy` | `rewind_pipeline.py` | GPU (SigLIP) |

**What changed in the merge**
- **Threshold:** the adaptive threshold now measures "normal" on the quieter half of the samples
  (`q_noise=0.5`, from v2). With all samples, a busy camera's median *is* motion: on the café video v1 marked
  people walking across the room as inactive. `q_noise=1.0` gives the old v1 behaviour. Section 6 shows both.
  Both changes move v1's baseline (109 keyframes on `meva_30min.mp4`): re-run it once to get the new number.
- **MOG2 runs on colour frames** (`mog2_color=True`, as v2 did). On greyscale, MOG2's shadow detector calls anything
  50-100% as bright as the background a shadow and drops it, e.g. a person in mid-dark clothes on a light floor.
  On a synthetic test, greyscale MOG2 missed a walker completely. `False` gives the old behaviour.
- **One stage 1:** v2 re-implemented diff + MOG2. Now every gate uses v1's code; the deep gate only adds a
  `score_deep` column and the fused vote is computed from the gates' scores.
- **Default gate = MOG2.** On the café ground truth, MOG2 (frame F1 0.507, event recall 0.744) and fused
  (0.500 / 0.785) were nearly tied, and fused needs a DINOv2 pass over the whole video. Switch with `GATE`.
- **Fair sampler comparison:** v1's keyframes are added to v2's sampler table (`v1_keyframes`).
- A model that fails to load is skipped with a warning, so the notebook also runs without a GPU, with fewer sections.
  SigLIP needs the `sentencepiece` package (its absence silently disabled SigLIP and 3 of 4 samplers in the first v2 run).

Edit the config cell, then *Run All*. Slow passes are cached in `outputs/<video>/cache/`.

In [ ]:
import base64, os, time, warnings
from datetime import datetime, timedelta
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")  # draw figures without a screen; they are embedded as images
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

import deep_gate                # optional DINOv2 gate (stage 1)
import evaluate as ev           # ground-truth checks
import keyframes as kfx         # stage 2
import motion_gating as mg      # stage 1
import rewind_pipeline as rp    # stages 3-4 (models load lazily, only when used)
import visualize as viz         # drawing helpers
import visualize_events as vev  # drawing helpers for stages 3-4
from video_utils import format_hms, video_info

if Path("/mnt/5T_storage").exists():  # team VM: Hugging Face models go to the 5T disk, not the nearly full root disk
    os.environ.setdefault("HF_HOME", "/mnt/5T_storage/hf_cache")
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
T0 = time.perf_counter()
TIMINGS = {}

def show_fig(fig):
    display(Image(viz.fig_png_bytes(fig)))
    plt.close(fig)

def show_img(img, quality=70):
    display(Image(viz.jpeg_bytes(img, quality)))

## Config

In [ ]:
VIDEO = Path("video/cafe_demo.mp4")   # any video
CAMERA_ID = "cafe01"                  # None = use the video file name
START_TIME = None                     # wall-clock time of frame 0, e.g. "2026-11-15T14:00:00"
GT_CSV = None                         # e.g. "gt_motion.csv" with start_s,end_s[,label]; None = skip GT checks

GATE = "mog2"          # gate that feeds keyframes and stages 3-4: "mog2" | "diff" | "fused" | "deep"
DEEP_GATE = True       # DINOv2 gate (GPU). Skipped with a warning if torch/transformers/model are missing
RUN_GPU_STAGES = True  # stages 3-4 (YOLO-World, SigLIP, X-CLIP, samplers). Same fallback

SCAN = dict(                # stage 1, slow pass (changing these re-decodes the video)
    sample_fps=5.0,         # frames analysed per second of video
    width=320,              # analysis width in pixels
    blur_ksize=5,           # blur to suppress pixel noise
    diff_threshold=15,      # frame differencing: grey-level change that counts as motion
    mog2_history=500,       # MOG2 memory in samples (500 = 100 s)
    mog2_var_threshold=16.0,
    mog2_learning_rate=-1.0,
    mog2_illum_ncc=0.7,     # lighting filter (None = off)
    mog2_color=True,        # MOG2 on colour frames (False = old v1 greyscale: mid-dark clothes count as "shadow")
    morph_kernel=3,         # noise clean-up size
    ignore_polygons=[],     # e.g. [[(0, 0), (1, 0), (1, 0.22), (0, 0.22)]] ignores the top 22% (sky)
    hash_grid=4, hist_bins=32,
)
SEGMENTS = dict(            # stage 1, cheap (no re-decoding)
    gates=["diff", "mog2"], # "deep" is added when DEEP_GATE works; "fused" is always computed
    threshold_mode="adaptive", k=3.0,
    q_noise=0.5,            # "normal" = quietest 50% of samples (1.0 = all samples, old v1)
    floor={"diff": 0.001, "mog2": 0.001, "deep": 0.004},
    fixed_threshold={"mog2": 0.004, "diff": 0.006},
    stop_ratio=0.5, smooth_s=1.0, merge_gap_s=3.0, min_duration_s=1.0, pad_s=2.0,
)
KEYFRAMES = dict(           # stage 2, cheap
    metric="phash", change_threshold=6.0, dedup_threshold=2.5,
    min_gap_s=3.0, max_gap_s=30.0, heartbeat_s=300.0,
)
# v2 settings for the deep gate and stages 3-4 (all keys: rewind_pipeline.DEFAULT_CONFIG)
V2 = rp.deep_update(rp.DEFAULT_CONFIG, {
    "scan": {"ignore_polygons": SCAN["ignore_polygons"]},
    "segments": {"heartbeat_s": KEYFRAMES["heartbeat_s"]},
    "sampling": {"primary": "kcenter", "export_jpgs": ["kcenter"]},
    "runtime": {"device": "auto", "mock_models": False},  # True = fake models: CPU plumbing test, numbers meaningless
})
NAIVE_FPS = 1.0             # baseline: "just take 1 frame per second"

TOP_N = 5                   # segments shown in detail
N_GIFS = 3
THUMB_W = 160
SWEEP_K = np.arange(1.0, 8.01, 0.5)

OUT = mg.output_dir(VIDEO)  # outputs/<video name>/
EV_OUT = OUT / "events"     # stages 3-4 write here
camera_id = CAMERA_ID or VIDEO.stem
t_start = datetime.fromisoformat(START_TIME) if START_TIME else None
models = rp.Models(V2)
print(f"video: {VIDEO} | outputs: {OUT} | device: {models.device}" + ("  (MOCK MODELS)" if models.mock else ""))

## 1. The video
12 frames spread over the video, to get a feel for the scene.

In [ ]:
info = video_info(VIDEO)
D = info.duration_s
display(pd.Series({"duration": format_hms(D), "fps": info.fps, "frames": info.frame_count,
                   "resolution": f"{info.width}x{info.height}"}).to_frame("value"))
show_img(viz.contact_sheet(VIDEO, info.fps, np.linspace(0, max(0, D - 1), 12), cols=4, width=240))

## 2. Stage 1: motion gating
One decode pass gives the diff and MOG2 scores (+ image fingerprints for the keyframes). The optional deep gate
runs DINOv2 on 2 frames/s. Then every gate gets its own adaptive threshold and segments, and the **fused** gate
averages the two strongest gates' scores, each divided by its own threshold (a soft "2 gates agree" vote).

In [ ]:
t0 = time.perf_counter()
scan = mg.scan_video(VIDEO, **SCAN, cache_dir=OUT / "cache")
TIMINGS["stage 1: decode + diff/MOG2 scores"] = time.perf_counter() - t0
scores, sig, st = scan["scores"], scan["sig"], scan["stats"]
print(f"{'loaded from cache' if st['from_cache'] else 'decoded'}: {st['frames_decoded']} frames, "
      f"{st['samples']} analysed (every {st['step']}th), decode pass {st['scan_s']:.1f} s")

gates_used = list(SEGMENTS["gates"])
if DEEP_GATE:
    t0 = time.perf_counter()
    deep, dst = deep_gate.scan_deep(VIDEO, scores, V2, models, cache_dir=OUT / "cache")
    TIMINGS["stage 1: deep gate (DINOv2)"] = time.perf_counter() - t0
    if deep is None:
        print("deep gate skipped:", dst.get("skipped"))
    else:
        scores["score_deep"] = deep
        gates_used.append("deep")
        print("deep gate:", "loaded from cache" if dst["from_cache"] else
              f"{dst['deep_frames']} frames in {dst['deep_s']:.1f} s")

t0 = time.perf_counter()
RESULTS = mg.segment_methods(scores, D, gates_used, **SEGMENTS)
mg.add_fused(scores, RESULTS, D, **SEGMENTS)
TIMINGS["stage 1: thresholds + segments"] = time.perf_counter() - t0
if GATE not in RESULTS:
    warnings.warn(f"gate {GATE!r} not available (deep gate off or failed?), using mog2")
    GATE = "mog2"
segments = RESULTS[GATE]["segments"]
SCALE, UNIT = (1, "x thr") if GATE == "fused" else (100, "%")   # how to print the chosen gate's scores
mg.save_results(OUT, scores, RESULTS, GATE)

display(pd.DataFrame({m: {"start threshold": r["start"] * (1 if m == "fused" else 100),
                          "unit": "x own thr" if m == "fused" else "% moving",
                          "segments": len(r["segments"]),
                          "active (% of video)": 100 * mg.active_fraction(r["segments"], D)}
                      for m, r in RESULTS.items()}).T.assign(used=lambda d: np.where(d.index == GATE, "<-", "")))
display(Markdown("**Agreement between gates** (IoU of active time: 1 = identical, 0 = no overlap)"))
display(ev.gate_agreement(RESULTS, scores.timestamp_s.to_numpy()).round(2))

## 3. Stage 2: keyframes
Computed for every gate (cheap) so they can be compared; JPGs are saved for the chosen gate only.

In [ ]:
t0 = time.perf_counter()
KF = {m: kfx.select_keyframes(scores, r["segments"], sig, score_col=r["col"], **KEYFRAMES) for m, r in RESULTS.items()}
TIMINGS["stage 2: select keyframes"] = time.perf_counter() - t0
t0 = time.perf_counter()
kf = kfx.export_keyframes(VIDEO, KF[GATE], OUT)
TIMINGS["stage 2: save JPGs"] = time.perf_counter() - t0
print(f"{len(kf)} keyframes ({GATE}): {kf['reason'].value_counts().to_dict()}")
kf.drop(columns="sample").head()

## 4. Timeline
Blue line = score. Above the red dashed line a segment *starts*, below the orange dotted line it *stops*.
Shaded = segments; ticks under the axis = keyframes coloured by reason (blue first · red peak · green change ·
purple maxgap · grey heartbeat). The fused score is a ratio: 1 = at the threshold.

In [ ]:
SMOOTH_WIN = max(1, round(SEGMENTS["smooth_s"] * st["effective_sample_fps"]))
names = {"diff": "frame differencing", "mog2": "MOG2 background subtraction",
         "deep": "DINOv2 deep gate (% of patches)", "fused": "fused soft vote of " + " + ".join(gates_used)}
fig, axes = plt.subplots(len(RESULTS), 1, figsize=(15, 3.1 * len(RESULTS)))
for ax, (m, r) in zip(np.atleast_1d(axes), RESULTS.items()):
    viz.plot_timeline(ax, scores, r["col"], r["segments"], r["start"], r["stop"],
                      1 if m == "fused" else SMOOTH_WIN, KF[m],
                      f"{m}: {names[m]}: {len(r['segments'])} segments, {mg.active_fraction(r['segments'], D):.1%} active, "
                      f"{len(KF[m])} keyframes" + ("  [used]" if m == GATE else ""),
                      scale=1 if m == "fused" else 100,
                      ylabel="ratio to threshold" if m == "fused" else "moving (%)")
    ax.legend(loc="upper left", fontsize=7, ncol=4)
fig.tight_layout()
show_fig(fig)

Average brightness over time. Lighting changes can fool MOG2; that's why the lighting filter exists.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 2.2))
ax.plot(scores.timestamp_s / 60, scores.mean_luma, lw=0.7, color="#555")
ax.set(xlabel="video time (min)", ylabel="mean grey level"); ax.grid(alpha=.25)
show_fig(fig)

## 5. Summary: how many frames would an expensive model have to look at?

In [ ]:
def summarize(m):
    segs, k = RESULTS[m]["segments"], KF[m]
    naive = int(D * NAIVE_FPS)
    act_min = mg.active_fraction(segs, D) * D / 60
    return {"segments": len(segs), "active (% of video)": 100 * mg.active_fraction(segs, D),
            "keyframes": len(k), "keyframes per minute": len(k) / (D / 60),
            "keyframes per active minute": (k.segment_id >= 0).sum() / max(act_min, 1e-9),
            f"naive {NAIVE_FPS:g} frame/s": naive, "reduction (x fewer frames)": naive / max(1, len(k))}

display(pd.DataFrame({m + (" [used]" if m == GATE else ""): summarize(m) for m in RESULTS}).round(1))
S = summarize(GATE)
display(Markdown(f"### {S['keyframes']} keyframes instead of {S[f'naive {NAIVE_FPS:g} frame/s']:,} "
                 f"(**{S['reduction (x fewer frames)']:.1f}x fewer**); {S['active (% of video)']:.1f}% of the video is active."))

## 6. What the threshold change did
Old v1 threshold (median + k·MAD of **all** samples) vs the merged one (of the **quietest** `q_noise` share).
On a mostly idle camera they should be close; on a busy one the old threshold sits above real activity.

In [ ]:
rows = []
kw = {x: SEGMENTS[x] for x in ("smooth_s", "merge_gap_s", "min_duration_s", "pad_s")}
for m in gates_used:
    for q in sorted({1.0, SEGMENTS["q_noise"]}, reverse=True):
        p = dict(SEGMENTS, q_noise=q)
        start, stop = mg.start_stop_thresholds(scores, m, **p)
        segs = mg.find_segments(scores, f"score_{m}", start_threshold=start, stop_threshold=stop, duration_s=D, **kw)
        rows.append({"gate": m, "threshold": "old: all samples" if q == 1.0 else f"new: quietest {q:.0%}",
                     "start threshold (%)": 100 * start, "active (% of video)": 100 * mg.active_fraction(segs, D),
                     "segments": len(segs),
                     "keyframes": len(kfx.select_keyframes(scores, segs, sig, score_col=f"score_{m}", **KEYFRAMES))})
display(pd.DataFrame(rows).set_index(["gate", "threshold"]).round(2))

## 7. Segments
All segments of the chosen gate, then the strongest ones: 6 frames spread over the segment, and the peak frame
with the MOG2 mask on top (**red** = moving pixels that count, **yellow** = blobs the lighting filter removed).

In [ ]:
segments.assign(peak=segments.peak_score * SCALE,
                keyframes=segments.segment_id.map(kf.segment_id.value_counts()).fillna(0).astype(int)
                )[["segment_id", "start_hms", "end_hms", "duration_s", "peak", "keyframes"]].rename(
                    columns={"peak": f"peak ({UNIT})"})

In [ ]:
t0 = time.perf_counter()
top_segs = segments.nlargest(TOP_N, "peak_score")
strip_idx = {s.segment_id: [int(round(t * info.fps)) for t in np.linspace(s.start_s, max(s.start_s, s.end_s - 0.5), 6)]
             for s in top_segs.itertuples()}
frames = viz.frames_at(VIDEO, [i for v in strip_idx.values() for i in v])
for s in top_segs.itertuples():
    strip = viz.grid([viz.label(viz.thumb(frames[i], THUMB_W), format_hms(i / info.fps)[3:10], scale=0.35)
                      for i in strip_idx[s.segment_id] if i in frames], cols=3, pad=2)
    peak, mask, removed = mg.foreground_mask_at(VIDEO, s.peak_frame_idx, scan["params"], "mog2")
    ov = viz.label(viz.thumb(viz.overlay_mask(peak, mask, removed=removed), 2 * THUMB_W + 2),
                   f"peak {format_hms(s.peak_time_s)[:10]}  {s.peak_score * SCALE:.2f}{UNIT}", scale=0.35)
    row = viz.grid([strip, ov], cols=2, pad=6)
    show_img(np.vstack([vev.header(f"segment {s.segment_id}: {s.start_hms[:8]}-{s.end_hms[:8]} ({s.duration_s:.0f} s)",
                                   row.shape[1]), row]))
TIMINGS["figures: top segments"] = time.perf_counter() - t0

### Animated previews
Low-res GIFs at 5x speed.

In [ ]:
html = []
for s in segments.nlargest(N_GIFS, "peak_score").sort_values("start_s").itertuples():
    p, size, speed = viz.segment_gif(VIDEO, info.fps, s.start_s, s.end_s, OUT / "gifs" / f"seg{s.segment_id:03d}.gif",
                                     caption=f"seg{s.segment_id:03d}")
    b64 = base64.b64encode(p.read_bytes()).decode()
    html.append(f'<figure style="display:inline-block;margin:4px"><img src="data:image/gif;base64,{b64}" width="256">'
                f'<figcaption style="font-size:11px">segment {s.segment_id}, {s.start_hms[:8]}-{s.end_hms[:8]}, '
                f'{speed:.0f}x, {size/1e3:.0f} KB</figcaption></figure>')
display(HTML("".join(html)) if html else Markdown("no segments"))

### All keyframes (chosen gate)
Timestamp and segment on top, reason at the bottom, border colour = reason. Full-size files: `outputs/<video>/keyframes/`.

In [ ]:
tiles = []
for r in kf.itertuples():
    col = viz.hex_to_bgr(viz.REASON_COLORS[r.reason])
    t = viz.label(viz.thumb(cv2.imread(str(OUT / "keyframes" / r.filename)), THUMB_W),
                  f"{r.timestamp_hms[:10]} " + ("idle" if r.segment_id < 0 else f"s{r.segment_id}"), scale=0.33)
    tiles.append(viz.border(viz.label(t, r.reason, pos="bottom", scale=0.33, bg=col), col, 2))
show_img(viz.grid(tiles, cols=8, pad=3), quality=60)

## 8. Sanity checks
**(a)** Random frames from *inactive* time: did the gate skip anything important?
**(b)** Suspicious segments (shorter than 8 s, or a peak barely above the threshold), worth a manual look.

In [ ]:
rng = np.random.default_rng(0)
t_all = scores.timestamp_s.to_numpy()
idle = np.flatnonzero(~mg.active_mask(t_all, segments))
print(f"(a) {idle.size / len(t_all):.0%} of samples are inactive." + (" Random examples:" if idle.size else ""))
if idle.size:
    pick = np.sort(rng.choice(idle, size=min(8, idle.size), replace=False))
    fr = viz.frames_at(VIDEO, scores.frame_idx.to_numpy()[pick])
    show_img(viz.grid([viz.label(viz.thumb(fr[scores.frame_idx.iat[i]], 240),
                                 f"{format_hms(t_all[i])[:8]}  {GATE} {scores[RESULTS[GATE]['col']].iat[i] * SCALE:.2f}{UNIT}",
                                 scale=0.38) for i in pick if scores.frame_idx.iat[i] in fr], cols=4))

In [ ]:
thr = RESULTS[GATE]["start"]
sus = segments[(segments.duration_s < 8) | (segments.peak_score < 1.5 * thr)].copy()
sus["why"] = np.where(sus.duration_s < 8, "short ", "") + np.where(sus.peak_score < 1.5 * thr, "low-peak", "")
print(f"(b) {len(sus)} suspicious segment(s)")
if len(sus):
    display(sus.assign(peak=sus.peak_score * SCALE)[["segment_id", "start_hms", "duration_s", "peak", "why"]])
    tiles = []
    for s in sus.itertuples():
        f, mk, rm = mg.foreground_mask_at(VIDEO, s.peak_frame_idx, scan["params"], "mog2")
        tiles.append(viz.label(viz.thumb(viz.overlay_mask(f, mk, removed=rm), 320),
                               f"seg {s.segment_id} peak {format_hms(s.peak_time_s)[:8]} ({s.why.strip()})", scale=0.38))
    show_img(viz.grid(tiles, cols=3))

## 9. Sensitivity sweep
How % active and keyframe count change with the threshold (adaptive `k` from 1 to 8). Uses the cached scores, so it's fast.

In [ ]:
t0 = time.perf_counter()
rows = []
for m in gates_used:
    for k_ in SWEEP_K:
        p = dict(SEGMENTS, k=k_, threshold_mode="adaptive")
        start, stop = mg.start_stop_thresholds(scores, m, **p)
        segs = mg.find_segments(scores, f"score_{m}", start_threshold=start, stop_threshold=stop, duration_s=D, **kw)
        rows.append({"gate": m, "k": k_, "threshold_pct": 100 * start, "active_pct": 100 * mg.active_fraction(segs, D),
                     "segments": len(segs),
                     "keyframes": len(kfx.select_keyframes(scores, segs, sig, score_col=f"score_{m}", **KEYFRAMES))})
sweep = pd.DataFrame(rows)
TIMINGS["sensitivity sweep"] = time.perf_counter() - t0
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for m, g in sweep.groupby("gate"):
    for ax, col in zip(axes, ("active_pct", "keyframes", "segments")):
        ax.plot(g.k, g[col], marker="o", ms=3, label=m)
for ax, t_ in zip(axes, ("% of video active", "keyframes", "segments")):
    ax.axvline(SEGMENTS["k"], color="#888", ls=":", lw=1)  # current setting
    ax.set(xlabel="k (threshold = median + k x MAD)", title=t_); ax.grid(alpha=.25)
axes[0].legend()
fig.tight_layout()
show_fig(fig)

## 10. Ground truth (only if `GT_CSV` is set)
Per gate: frame precision / recall / F1 and **event recall** (share of labelled events a segment covers).
Per keyframe set: share of labelled events that got **at least one keyframe**. That is what matters downstream:
an event without a keyframe is never captioned or indexed. Stage 4 adds the v2 samplers to this table.

In [ ]:
gt = ev.load_gt(GT_CSV) if GT_CSV else None
if gt is not None:
    g_tab = ev.gate_table(gt, RESULTS, scores.timestamp_s.to_numpy(), D)
    k_tab = ev.keyframe_hits(gt, {f"v1 keyframes ({m})": KF[m].timestamp_s for m in RESULTS})
    g_tab.to_csv(OUT / "gt_eval_gates.csv"); k_tab.to_csv(OUT / "gt_eval_keyframes.csv")
    print(f"{len(gt)} ground-truth events covering {100 * mg.active_fraction(gt.assign(duration_s=gt.end_s - gt.start_s), D):.1f}% of the video")
    display(g_tab.round(3)); display(k_tab.round(3))
else:
    print("GT_CSV not set, skipped")

## 11. Stage 3: events (GPU, optional)
A second decode inside the chosen gate's segments at 4 frames/s: YOLO-World detector + ByteTrack tracker
(rules: appears, loitering, running, object left, crowd), SigLIP zero-shot frame tags (its embeddings are also
the search index), X-CLIP zero-shot 8-frame clip tags. Methods whose model is missing are skipped.

Known limits from the first café run: track IDs switch a lot in crowds (many "appears" events are the same person),
"crowd" (≥5 people) and "loitering" fire almost all the time in a busy lobby, and X-CLIP's softmax over 13 prompts
rarely passes its 0.35 threshold. The rules were written for outdoor CCTV and need tuning per scene.

In [ ]:
GPU = RUN_GPU_STAGES
if GPU:
    EV_OUT.mkdir(parents=True, exist_ok=True)
    vinfo = rp.video_info(VIDEO)   # v2's VideoInfo (same numbers, different class)
    t0 = time.perf_counter()
    feat = rp.extract_event_features(VIDEO, vinfo, segments, V2, models)
    meta, E = feat["meta"], feat["emb"]
    use = V2["events"]["methods"]
    ev_tr, track_sum = rp.track_events(feat["tracks"], V2) if "tracks" in use and len(feat["tracks"]) \
        else (pd.DataFrame(), pd.DataFrame())
    ev_sg, sg_extra = rp.siglip_events(feat, V2, models) if "siglip" in use else (pd.DataFrame(), None)
    ev_xc = rp.xclip_events(feat, V2) if "xclip" in use else pd.DataFrame()
    events = rp.merge_events([ev_tr, ev_sg, ev_xc], V2["events"]["merge_gap_s"])
    scores["score_ratio"] = scores[RESULTS[GATE]["col"]] / RESULTS[GATE]["start"]  # motion part of importance
    imp = rp.importance_curve(meta, scores, "score_ratio", events, track_sum, V2)
    anchors = rp.event_anchors(meta, events, track_sum, V2)
    TIMINGS["stage 3: events"] = time.perf_counter() - t0
    meta["importance"] = imp
    if sg_extra is not None:
        meta["siglip_tag"] = sg_extra["tags"]
    if t_start is not None and len(events):
        events["start_wallclock"] = [(t_start + timedelta(seconds=float(s))).isoformat() for s in events.start_s]
    events.insert(1, "camera_id", camera_id)
    events.to_csv(EV_OUT / "events.csv", index=False)
    feat["tracks"].to_csv(EV_OUT / "tracks_raw.csv", index=False)
    track_sum.to_csv(EV_OUT / "tracks.csv", index=False)
    feat["xclip"].to_csv(EV_OUT / "xclip_windows.csv", index=False)
    meta.drop(columns="phash").to_csv(EV_OUT / "candidate_frames.csv", index=False)
    by_src = events.source.value_counts().to_dict() if len(events) else {}
    print(f"{len(events)} events {by_src}, {int((events.support >= 2).sum()) if len(events) else 0} confirmed by 2+ methods, "
          f"{len(track_sum)} tracks, {len(meta)} candidate frames, embeddings {E.shape}")
    if feat["stats"]["skipped_methods"]:
        display(Markdown("**Skipped (model failed to load):**"))
        display(pd.Series(feat["stats"]["skipped_methods"], name="error").to_frame())
    display(Markdown("**Merged events**")); display(events.head(30))
    display(Markdown(f"**Tracks** ({len(track_sum)})")); display(track_sum.head(15))
else:
    print("RUN_GPU_STAGES is off: stages 3-4 skipped")

**Look at it.** What the detector and tracker see: boxes coloured by track, SigLIP tag under each frame.

In [ ]:
if GPU:
    tracks = feat["tracks"]
    fw = V2["events"]["frame_width"]
    for seg in segments.nlargest(TOP_N, "peak_score").sort_values("start_s").itertuples():
        rows = meta[meta.segment_id == seg.segment_id]
        if rows.empty:
            continue
        pick = rows.iloc[np.linspace(0, len(rows) - 1, min(6, len(rows))).round().astype(int)]
        fr = viz.frames_at(VIDEO, pick.frame_idx)
        tiles = []
        for r in pick.itertuples():
            if r.frame_idx not in fr:
                continue
            img = viz.thumb(vev.draw_boxes(viz.thumb(fr[r.frame_idx], fw), vev.boxes_for(tracks, r.frame_idx)), 2 * THUMB_W)
            img = viz.label(img, format_hms(r.timestamp_s)[3:10], scale=0.35)
            tiles.append(viz.label(img, f"SigLIP: {getattr(r, 'siglip_tag', '') or '-'}", pos="bottom", scale=0.35,
                                   bg=(60, 60, 60)))
        g = viz.grid(tiles, cols=3, pad=2)
        show_img(np.vstack([vev.header(f"segment {seg.segment_id}: {seg.start_hms[:8]}-{seg.end_hms[:8]}", g.shape[1]), g]))

**Look at it.** Event gallery (peak frame of each event; border: grey = 1 method, orange = 2, green = 3 agree) and when each event happens.

In [ ]:
if GPU:
    SUPPORT_COL = {1: (160, 160, 160), 2: (0, 165, 255), 3: (0, 180, 0)}
    ev_show = events.sort_values(["support", "score"], ascending=False).head(24) if len(events) else events
    if ev_show.empty:
        print("no events")
    else:
        fi = {}
        for r in ev_show.itertuples():
            m_ = meta[meta.segment_id == r.segment_id]
            if len(m_):
                fi[r.event_id] = int(m_.frame_idx.iat[(m_.timestamp_s - r.peak_s).abs().argmin()])
        fr = viz.frames_at(VIDEO, fi.values())
        tiles = []
        for r in ev_show.itertuples():
            i = fi.get(r.event_id)
            if i not in fr:
                continue
            img = viz.thumb(vev.draw_boxes(viz.thumb(fr[i], fw), vev.boxes_for(tracks, i)), 2 * THUMB_W)
            img = viz.label(img, r.key, scale=0.42)
            img = viz.label(img, f"{r.supported_by} {r.score:.2f} @ {format_hms(r.peak_s)[3:10]}", pos="bottom",
                            scale=0.35, bg=(60, 60, 60))
            tiles.append(viz.border(img, SUPPORT_COL.get(min(r.support, 3)), 3))
        show_img(viz.grid(tiles, cols=4), quality=70)

        keys = sorted(events.key.unique())
        fig, ax = plt.subplots(figsize=(15, 0.35 * len(keys) + 1.2))
        colors = {"tracks": "C0", "siglip": "C2", "xclip": "C3"}
        for r in events.itertuples():
            y = keys.index(r.key) + {"tracks": -0.25, "siglip": 0, "xclip": 0.25}.get(r.source, 0)
            ax.plot([r.start_s / 60, r.end_s / 60], [y, y], lw=4, color=colors.get(r.source, "k"), solid_capstyle="butt")
        for s_ in segments.itertuples():
            ax.axvspan(s_.start_s / 60, s_.end_s / 60, color="C1", alpha=0.1)
        ax.set_yticks(range(len(keys)), keys, fontsize=8)
        ax.set_xlim(0, D / 60); ax.set_xlabel("video time (min)"); ax.grid(alpha=0.25)
        ax.set_title("events: blue = tracks, green = SigLIP, red = X-CLIP (shaded = motion segments)", fontsize=9, loc="left")
        show_fig(fig)

**Look at it.** SigLIP heatmap: how strongly each event prompt beats the "nothing happening" prompts, frame by frame.

In [ ]:
if GPU:
    if sg_extra is not None and len(meta):
        act_rows = np.flatnonzero(meta.segment_id.to_numpy() >= 0)
        if act_rows.size:
            Mg = sg_extra["margin"][act_rows].T
            thr_ = V2["events"]["siglip_margin"]
            lim = max(1e-6, float(np.abs(Mg - thr_).max()))
            fig, ax = plt.subplots(figsize=(15, 0.3 * len(rp.EVENT_BANK) + 1.2))
            im = ax.imshow(Mg - thr_, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, interpolation="nearest")
            ax.set_yticks(range(len(rp.EVENT_BANK)), [b["key"] for b in rp.EVENT_BANK], fontsize=8)
            tk = np.linspace(0, len(act_rows) - 1, min(12, len(act_rows))).astype(int)
            ax.set_xticks(tk, [format_hms(meta.timestamp_s.iat[act_rows[i]])[3:8] for i in tk], fontsize=7)
            ax.set_xlabel("candidate frames inside segments (time)")
            ax.set_title("SigLIP margin minus threshold (red = event detected)", fontsize=9, loc="left")
            fig.colorbar(im, ax=ax, fraction=0.02)
            show_fig(fig)
    else:
        print("SigLIP not available (check that sentencepiece is installed)")

**Look at it.** Animated previews with tracker boxes, 5x speed.

In [ ]:
if GPU:
    html = []
    for seg in segments.nlargest(N_GIFS, "peak_score").sort_values("start_s").itertuples():
        rows = meta[meta.segment_id == seg.segment_id]
        if len(rows) < 2:
            continue
        fr = viz.frames_at(VIDEO, rows.frame_idx)
        frames_ = [vev.draw_boxes(viz.thumb(fr[i], fw), vev.boxes_for(tracks, i)) for i in rows.frame_idx if i in fr]
        html.append(vev.embed_gif(frames_, EV_OUT / "gifs" / f"seg{seg.segment_id:03d}.gif", V2["events"]["event_fps"] * 5,
                                  f"segment {seg.segment_id}, {seg.start_hms[:8]}-{seg.end_hms[:8]}, 5x"))
    display(HTML("".join(html)) if html else Markdown("no segments"))

## 12. Stage 4: which frames go to the captioner / index (GPU, optional)
Every sampler picks from the **same** candidate frames (stage 3's 4 frames/s inside segments):
`phash` (v1-style, one global hash), `novelty` (new in SigLIP space), `kcenter` (importance-weighted coverage,
primary), `importance` (sampled along the importance curve), plus **`v1_keyframes`**: this notebook's stage 2
keyframes moved to the nearest candidate frame, so v1 and v2 are scored the same way.

Scores: coverage (how close every candidate is to a kept frame, in SigLIP space), redundancy (how similar kept
frames are to each other), event / track recall (share of stage 3 events / tracks that got a keyframe).
Event and track recall are only as good as stage 3's events.

In [ ]:
def nearest_candidates(meta, kf):
    # candidate-row index nearest to each keyframe, within the same segment (heartbeats: idle rows)
    t, sid = meta.timestamp_s.to_numpy(), meta.segment_id.to_numpy()
    out = set()
    for r in kf.itertuples():
        m_ = np.flatnonzero(sid == r.segment_id) if r.segment_id >= 0 else np.flatnonzero(sid < 0)
        if m_.size:
            out.add(int(m_[np.argmin(np.abs(t[m_] - r.timestamp_s))]))
    return sorted(out)

if GPU:
    t0 = time.perf_counter()
    c = V2["sampling"]
    keeps = {}
    for name in c["methods"]:
        if name != "phash" and E.shape[1] == 0:
            print(f"sampler {name}: skipped (no SigLIP embeddings)")
            continue
        keeps[name] = rp.SAMPLERS[name](meta, c, E=E, imp=imp, anchors=anchors)
    keeps["v1_keyframes"] = nearest_candidates(meta, KF[GATE])
    evals = {n: rp.evaluate_sampling(meta, E, k, events, track_sum, vinfo, c) for n, k in keeps.items()}
    TIMINGS["stage 4: samplers"] = time.perf_counter() - t0
    psamp = c["primary"] if c["primary"] in keeps else next(iter(keeps))
    ev_df = pd.DataFrame(evals).T
    ev_df.to_csv(EV_OUT / "sampling_comparison.csv")
    display(ev_df.round(3).assign(primary=lambda d: np.where(d.index == psamp, "<-", "")))

**Look at it.** The same segment through each sampler (first 10 frames each). Look for near-identical frames (redundant) and moments with no frame (missed).

In [ ]:
if GPU:
    mt = meta.timestamp_s.to_numpy()
    for seg in segments.nlargest(min(3, TOP_N), "peak_score").sort_values("start_s").itertuples():
        sel = {n: [i for i in k if meta.segment_id.iat[i] == seg.segment_id] for n, k in keeps.items()}
        fr = viz.frames_at(VIDEO, {meta.frame_idx.iat[i] for k in sel.values() for i in k[:10]})
        rows_ = []
        for name, k in sel.items():
            tiles = [viz.label(viz.thumb(fr[meta.frame_idx.iat[i]], 120), format_hms(mt[i])[3:10], scale=0.32)
                     for i in k[:10] if meta.frame_idx.iat[i] in fr]
            strip = viz.grid(tiles, cols=10, pad=2) if tiles else np.full((40, 40, 3), 255, np.uint8)
            rows_.append(np.hstack([vev.name_tile(name, f"{len(k)} frames", strip.shape[0]), strip]))
        width = max(r.shape[1] for r in rows_)
        rows_ = [cv2.copyMakeBorder(r, 0, 4, 0, width - r.shape[1], cv2.BORDER_CONSTANT, value=(255, 255, 255)) for r in rows_]
        show_img(np.vstack([vev.header(f"segment {seg.segment_id}: {seg.start_hms[:8]}-{seg.end_hms[:8]} "
                                       f"({seg.duration_s:.0f} s)", width)] + rows_))

In [ ]:
if GPU:
    cols_ = ["keyframes", "coverage_mean", "coverage_p10", "redundancy_mean", "event_recall", "track_recall"]
    titles = {"keyframes": "keyframes (lower = cheaper)", "coverage_mean": "coverage mean (higher = better)",
              "coverage_p10": "worst-10% coverage (higher = better)", "redundancy_mean": "redundancy (lower = better)",
              "event_recall": "events with a keyframe", "track_recall": "tracks with a keyframe"}
    fig, axes = plt.subplots(1, len(cols_), figsize=(16, 2.8))
    for ax, c_ in zip(axes, cols_):
        v = ev_df[c_].astype(float)
        ax.bar(v.index, v.values, color=["C1" if n == psamp else "C0" for n in v.index])
        ax.set_title(titles[c_], fontsize=8); ax.tick_params(axis="x", labelsize=7, rotation=30); ax.grid(alpha=0.25, axis="y")
    fig.suptitle(f"orange = primary sampler ({psamp})", fontsize=9)
    fig.tight_layout()
    show_fig(fig)
    if gt is not None:
        display(Markdown("**Ground truth: labelled events with at least one keyframe, per sampler**"))
        s_tab = ev.keyframe_hits(gt, {n: mt[np.array(k, int)] for n, k in keeps.items()})
        s_tab.to_csv(EV_OUT / "gt_eval_sampling.csv")
        display(s_tab.round(3))

### Hand-off to captioning / indexing
For the primary sampler: `events/index_records.jsonl` (one record per keyframe: camera, time, wall-clock,
events, objects, SigLIP tag, `embedding_row`) and `events/index_embeddings.npy` (SigLIP vectors, 768 numbers
each for `siglip-base-patch16-224`). Plus a CSV of every sampler's keyframes and JPGs for `export_jpgs`.

In [ ]:
if GPU:
    for name, keep in keeps.items():
        kf_s = meta.loc[keep].assign(camera_id=camera_id, sampler=name)
        kf_s["timestamp_hms"] = kf_s.timestamp_s.map(format_hms)
        if name in c["export_jpgs"]:
            kf_s["filename"] = rp.export_keyframes(VIDEO, vinfo, meta, keep, EV_OUT, name, camera_id)
        kf_s.drop(columns="phash").to_csv(EV_OUT / f"keyframes_{name}.csv")
    rec_path = rp.write_index_records(EV_OUT, meta, keeps[psamp], E, events, track_sum, camera_id, VIDEO.name, t_start)
    recs = pd.read_json(rec_path, lines=True)
    print(f"{len(recs)} index records from sampler '{psamp}' -> {rec_path}; embeddings {E[keeps[psamp]].shape if E.shape[1] else 'none'}")
    display(recs.head(10))

In [ ]:
if GPU and len(recs):
    # every keyframe of the primary sampler. Top: time, segment. Bottom: events (or SigLIP tag). Border = importance.
    fr = viz.frames_at(VIDEO, recs.frame_idx)
    cmap = matplotlib.colormaps["viridis"]
    tiles = []
    for r in recs.itertuples():
        if r.frame_idx not in fr:
            continue
        img = viz.thumb(vev.draw_boxes(viz.thumb(fr[r.frame_idx], fw), vev.boxes_for(tracks, r.frame_idx)), THUMB_W)
        img = viz.label(img, f"{format_hms(r.timestamp_s)[3:10]} " + ("idle" if r.segment_id < 0 else f"s{r.segment_id}"), scale=0.33)
        txt = ", ".join(r.events) if len(r.events) else (r.siglip_tag or "-")
        img = viz.label(img, txt[:28], pos="bottom", scale=0.32, bg=(60, 60, 60))
        col = tuple(int(x) for x in (np.array(cmap(float(r.importance))[:3])[::-1] * 255))
        tiles.append(viz.border(img, col, 3))
    show_img(viz.grid(tiles, cols=8), quality=65)

## 13. Everything on one timeline

In [ ]:
if GPU:
    rp.plot_timeline(EV_OUT / "timeline.png", scores, RESULTS, GATE, events,
                     {n: meta.timestamp_s.to_numpy()[np.array(k, int)] for n, k in keeps.items()}, vinfo)
    display(Image(filename=str(EV_OUT / "timeline.png")))
else:
    print("needs the GPU stages")

## Run times

In [ ]:
TIMINGS["whole notebook"] = time.perf_counter() - T0
display(pd.Series(TIMINGS, name="seconds").round(2).to_frame())